# CONTINUAL LEARNING ENABLED VISION FOUNDATION MODEL FOR SHIPPING CONTAINER DEFECT SEGMENTATION

```
Student: Feras Hamid Banday
```

### PROJECT OVERVIEW

This notebook implements a continual learning framework for container defect
segmentation using DINOv3 as the backbone and various CL strategies from
Mammoth library.

### Architecture:
- **Backbone:** DINOv3 (frozen, pre-trained)
- **Decoder:** Lightweight segmentation head (trainable)
- **CL Strategies:** ER, ER-ACE, DER/DER++

### Goals:
1. Train baseline model (all tasks at once)
2. Implement task-incremental learning
3. Compare CL strategies vs catastrophic forgetting
4. Benchmark performance: Average Accuracy, Backward Transfer, Forward Transfer

## **SECTION 1: CONFIGURATION**

All hyperparameters and paths defined here for easy modification.

In [ ]:
# Dataset Selection
DATASET_TYPE = "NEW"  # Options: "OLD", "NEW"

# Paths
DATA_DIR = "/content/drive/MyDrive"
OLD_ZIP_PATH = f"{DATA_DIR}/binary_mask_dataset.zip"
NEW_ZIP_PATH = f"{DATA_DIR}/coco_dataset.zip"

if DATASET_TYPE == "OLD":
    ZIP_PATH = OLD_ZIP_PATH
    DATASET_ROOT = "/content/old_dataset"
    NUM_CLASSES = 2
else:
    ZIP_PATH = NEW_ZIP_PATH
    DATASET_ROOT = "/content/coco_dataset"
    NUM_CLASSES = 6 # Background + 5 active COCO classes

TRAIN_DIR = f"{DATASET_ROOT}/train"
VALID_DIR = f"{DATASET_ROOT}/valid"
SAVE_DIR = "/content/drive/MyDrive/FYP_Models"
RESULTS_DIR = "/content/drive/MyDrive/FYP_Results"

import torch
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Model Configuration
BACKBONE = "vitl16"        
IMAGE_SIZE = 768            
PATCH_SIZE = 16
DECODER_HIDDEN_DIM = 256
DROPOUT = 0.1               
UNFREEZE_LAST_N_BLOCKS = 0  

# Training Configuration
BATCH_SIZE = 8
LEARNING_RATE = 5e-4        
NUM_EPOCHS = 10             
SUBSET_SIZE = None          

# Continual Learning Configuration
NUM_TASKS = 5              
MEMORY_SIZE = 500           
MER_BETA = 0.1              
MER_GAMMA = 0.1             
MER_BATCH_NUM = 1

# SI Hyperparameters (aimagelab/mammoth architecture)
SI_C = 0.5                  # penalty weight - higher = more protection, less plasticity
SI_XI = 1.0                 # damping term - prevents division by zero

# LwF Hyperparameters
LWF_ALPHA = 0.5             # distillation loss weight
LWF_TEMPERATURE = 2.0       # softmax temperature

# SLCA Hyperparameters
SLCA_CA_EPOCHS = 5          # alignment epochs
SLCA_LOGIT_NORM = 0.1       # logit L2 norm weight

# Task Splitting Options
IGNORE_COMPOSITE = False    
ALLOW_REUSE = False         

print(f"Configuration loaded for {DATASET_TYPE} dataset")


## **SECTION 2: ENVIRONMENT SETUP**

Install dependencies and prepare environment

In [ ]:
# Install Dependencies
!pip install -q torch torchvision pillow numpy matplotlib tqdm transformers huggingface_hub
print("Dependencies installed")

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# HuggingFace Authentication
from huggingface_hub import login
from google.colab import userdata
login(token=userdata.get('HF_TOKEN'))
print("HuggingFace authenticated")

In [ ]:
# Extract Dataset
import os
import zipfile
from pathlib import Path

os.makedirs(SAVE_DIR, exist_ok=True)

if not os.path.exists(DATASET_ROOT):
    print(f"Extracting {DATASET_TYPE} dataset to {DATASET_ROOT}")
    with zipfile.ZipFile(ZIP_PATH, 'r') as zip_ref:
        zip_ref.extractall(DATASET_ROOT)
    print("Dataset extracted")
else:
    print(f"{DATASET_TYPE} Dataset already extracted")

# Verification
def count_images(directory):
    return len(list(Path(directory).rglob("*.jpg")))

train_count = count_images(TRAIN_DIR)
valid_count = count_images(VALID_DIR)
print(f"Dataset verified: {train_count} train, {valid_count} valid images")

## **SECTION 3: DATA PIPELINE**

**Dataset Class**

Loads container images and binary defect masks.
```
  -  Resizes to 768x768 (compatible with DINOv3 patches)
  -  Applies ImageNet normalization
  -  Handles [0,1] mask values properly
```






In [ ]:
import os
import json
import numpy as np
import torch
from torch.utils.data import Dataset
from PIL import Image, ImageDraw
import torchvision.transforms as T
from typing import Tuple, Optional, List, Dict
from pathlib import Path


class ContainerDefectDataset(Dataset):
    """Dataset for shipping container defect segmentation (supports both Old and New formats)"""

    def __init__(
        self,
        root_dir: str | Path,
        image_size: int = IMAGE_SIZE,
        patch_size: int = PATCH_SIZE,
        subset_size: Optional[int] = None
    ):
        self.root_dir = Path(root_dir)
        assert image_size % patch_size == 0, "Image size must be divisible by patch size"
        self.image_size = image_size
        self.patch_size = patch_size
        self.dataset_type = DATASET_TYPE

        # Data collection
        self.image_files = []
        self.coco_data = None
        self.image_id_to_annotations = {}

        if self.dataset_type == "OLD":
            self._init_old_format(subset_size)
        else:
            self._init_coco_format(subset_size)

        # Image transform: resize + normalize
        self.image_transform = T.Compose([
            T.Resize((image_size, image_size)),
            T.ToTensor(),
            T.Normalize(
                mean=(0.485, 0.456, 0.406),
                std=(0.229, 0.224, 0.225),
            ),
        ])

        # Mask transform for OLD format: resize + scale [0,1]->[0,1]
        self.mask_transform = T.Compose([
            T.Resize(
                (image_size, image_size),
                interpolation=T.InterpolationMode.NEAREST
            ),
            T.ToTensor(),
            T.Lambda(lambda x: x * 255.0), # masks stored as [0,1] not [0,255]
        ])

        print(f"Loaded {len(self.image_files)} samples from {self.root_dir} ({self.dataset_type} mode)")

    def _init_old_format(self, subset_size):
        if not self.root_dir.exists():
            print(f"Warning: Directory {self.root_dir} does not exist.")
            return
        
        all_files = sorted(os.listdir(self.root_dir))
        self.image_files = [f for f in all_files if f.lower().endswith(".jpg")]
        
        if subset_size is not None:
            self.image_files = self.image_files[:subset_size]
        
        self._verify_masks()

    def _init_coco_format(self, subset_size):
        anno_path = self.root_dir / "_annotations.coco.json"
        if not anno_path.exists():
            # Try searching in subdirectories if extracted differently
            found_annos = list(self.root_dir.glob("**/_annotations.coco.json"))
            if found_annos:
                anno_path = found_annos[0]
            else:
                print(f"Warning: COCO annotations not found at {anno_path}")
                return

        with open(anno_path, 'r') as f:
            self.coco_data = json.load(f)

        # Map image_id to annotations
        from collections import defaultdict
        self.image_id_to_annotations = defaultdict(list)
        for anno in self.coco_data['annotations']:
            self.image_id_to_annotations[anno['image_id']].append(anno)

        # Filter images that actually exist and have annotations (optional)
        all_images = self.coco_data['images']
        if subset_size is not None:
            all_images = all_images[:subset_size]

        self.image_files = all_images # List of dicts in COCO mode

    def _verify_masks(self):
        """Verify all masks exist (for OLD format)"""
        missing = []
        for img in self.image_files:
            base = os.path.splitext(img)[0]
            mask_path = self.root_dir / f"{base}_mask.png"
            if not mask_path.exists():
                missing.append(str(mask_path))
        if missing:
            raise FileNotFoundError(f"Missing {len(missing)} masks")

    def __len__(self) -> int:
        return len(self.image_files)

    def __getitem__(self, idx: int) -> Tuple[torch.Tensor, torch.Tensor, str]:
        if self.dataset_type == "OLD":
            return self._get_old_item(idx)
        else:
            return self._get_coco_item(idx)

    def _get_old_item(self, idx):
        img_name = self.image_files[idx]
        img_path = self.root_dir / img_name
        mask_path = self.root_dir / f"{os.path.splitext(img_name)[0]}_mask.png"

        image = Image.open(img_path).convert("RGB")
        mask = Image.open(mask_path).convert("L")

        image = self.image_transform(image)
        mask = self.mask_transform(mask)
        mask = (mask > 0.5).float() # binarize (Background=0, Defect=1)

        return image, mask, img_name

    def _get_coco_item(self, idx):
        img_info = self.image_files[idx]
        img_id = img_info['id']
        img_name = img_info['file_name']
        
        # Paths might be relative to JSON or root
        img_path = self.root_dir / img_name
        if not img_path.exists():
            # Handle potential nested folder structure
            img_path = Path(next(self.root_dir.glob(f"**/{img_name}")))

        image = Image.open(img_path).convert("RGB")
        w, h = image.size
        
        # Create multi-class mask
        # 0 is Background, 1-6 are categories
        mask = Image.new('L', (w, h), 0)
        draw = ImageDraw.Draw(mask)
        
        annotations = self.image_id_to_annotations.get(img_id, [])
        for anno in annotations:
            cat_id = anno['category_id']
            # Roboflow category IDs 1-5 map to labels 1-5
            # Background is 0
            label = cat_id
            
            segmentation = anno['segmentation']
            if isinstance(segmentation, list):
                for poly in segmentation:
                    if len(poly) >= 4: # Need at least 2 points
                        draw.polygon(poly, fill=label)
            # RLE not handled here for simplicity, but could be added with pycocotools

        image = self.image_transform(image)
        # Resize mask with NEAREST to preserve class labels
        mask = mask.resize((self.image_size, self.image_size), resample=Image.NEAREST)
        mask_tensor = torch.from_numpy(np.array(mask)).unsqueeze(0).float()
        
        return image, mask_tensor, img_name

    def patch_grid(self) -> Tuple[int, int]:
        """Returns (H_patches, W_patches)"""
        n = self.image_size // self.patch_size
        return n, n


## **SECTION 4: MODEL ARCHITECTURE**


**DINOv3 Backbone**
*   Self-supervised vision transformer
*   Extracts dense patch features (48x48 grid for 768px images)
*   Frozen during training (only decoder is trainer)

<br>

**Segmentation Head**
*   Lightweight CNN decoder
*   Progressive upsampling: 48->96->192->384->768
*   Trainable parameters: ~2.7M (vs 303M frozen in DINOv3)


In [ ]:
import torch.nn as nn
from transformers import AutoModel




###**DINOv3 Feature Extractor**




In [ ]:
import torch
import torch.nn as nn
from transformers import AutoModel


class DINOv3FeatureExtractor(nn.Module):
    """Extracts dense features from DINOv3 backbone"""

    MODEL_REGISTRY = {
        "vits16": "facebook/dinov3-vits16-pretrain-lvd1689m",
        "vitsp16": "facebook/dinov3-vits16plus-pretrain-lvd1689m",
        "vitb16": "facebook/dinov3-vitb16-pretrain-lvd1689m",
        "vitl16": "facebook/dinov3-vitl16-pretrain-lvd1689m",
        "vithp16": "facebook/dinov3-vith16plus-pretrain-lvd1689m",
    }

    FEATURE_DIMS = {
        "vits16": 384,
        "vitsp16": 384,
        "vitb16": 768,
        "vitl16": 1024,
        "vithp16": 1280,
    }

    def __init__(
        self,
        model_name: str = BACKBONE,
        patch_size: int = PATCH_SIZE,
        freeze: bool = True,
        unfreeze_last_n_blocks: int = 0,
    ):
        super().__init__()
        self.model_name = model_name
        self.patch_size = patch_size
        self.feature_dim = self.FEATURE_DIMS[model_name]

        # Load pre-trained model
        self.backbone = AutoModel.from_pretrained(self.MODEL_REGISTRY[model_name])

        # Freeze everything first
        for param in self.backbone.parameters():
            param.requires_grad = False

        # Selectively unfreeze last N blocks and final layernorm
        if unfreeze_last_n_blocks > 0:
            blocks = self.backbone.layer
            for block in blocks[-unfreeze_last_n_blocks:]:
                for param in block.parameters():
                    param.requires_grad = True
            for param in self.backbone.norm.parameters():
                param.requires_grad = True

        if freeze and unfreeze_last_n_blocks == 0:
            self.backbone.eval()

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        B, C, H, W = x.shape
        H_patches, W_patches = H // self.patch_size, W // self.patch_size

        outputs = self.backbone(
            x,
            output_hidden_states=True,
            return_dict=True
        )

        # Extract spatial patch tokens (skip CLS + register tokens)
        last_layer = outputs.hidden_states[-1]
        num_patch_tokens = H_patches * W_patches
        patch_tokens = last_layer[:, 1:1+num_patch_tokens, :]

        # Reshape to spatial grid
        features = patch_tokens.reshape(B, H_patches, W_patches, self.feature_dim)
        return features.permute(0, 3, 1, 2)

    def get_feature_dim(self) -> int:
        return self.feature_dim


###**Segmentation Head (Decoder)**

In [ ]:
import torch
import torch.nn as nn


class SegmentationHead(nn.Module):
    """Upsamples features to pixel-wise predictions"""

    def __init__(
        self,
        in_channels: int,
        num_classes: int = NUM_CLASSES,
        hidden_dim: int = DECODER_HIDDEN_DIM,
        dropout: float = DROPOUT,
    ):
        super().__init__()

        # 48→96→192→384→768 progressive upsampling
        self.layer1 = self._make_layer(in_channels, hidden_dim, dropout)
        self.layer2 = self._make_layer(hidden_dim, hidden_dim // 2, dropout)
        self.layer3 = self._make_layer(hidden_dim // 2, hidden_dim // 4, dropout)
        self.layer4 = self._make_layer(hidden_dim // 4, hidden_dim // 8, dropout)
        self.classifier = nn.Conv2d(hidden_dim // 8, num_classes, kernel_size=1)

    def _make_layer(self, in_ch, out_ch, dropout):
        return nn.Sequential(
            nn.Conv2d(in_ch, out_ch, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.Dropout2d(dropout),
            nn.Upsample(scale_factor=2, mode='bilinear', align_corners=False),
        )

    def forward(self, features: torch.Tensor) -> torch.Tensor:
        x = self.layer1(features)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)
        return self.classifier(x)


### **Complete Model**

In [ ]:
import torch
import torch.nn as nn




class ContainerDefectSegmenter(nn.Module):
    """Complete segmentation model: DINOv3 + Decoder"""

    def __init__(
        self,
        backbone_name: str = BACKBONE,
        num_classes: int = NUM_CLASSES,
        freeze_backbone: bool = True,
        decoder_hidden_dim: int = DECODER_HIDDEN_DIM,
        unfreeze_last_n_blocks: int = 0,
    ):
        super().__init__()

        self.backbone = DINOv3FeatureExtractor(
            model_name=backbone_name,
            freeze=freeze_backbone,
            unfreeze_last_n_blocks=unfreeze_last_n_blocks,
        )

        # Segmentation head (decoder)
        self.head = SegmentationHead(
            self.backbone.get_feature_dim(),
            num_classes,
            decoder_hidden_dim
        )

        self.num_classes = num_classes

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        features = self.backbone(x)
        return self.head(features)

    def predict(self, x: torch.Tensor) -> torch.Tensor:
        return torch.argmax(self.forward(x), dim=1)


## **SECTION 5: TRAINING UTILITIES**

**Training & Evaluation**

Metrics:
*   **IoU (Intersection over Union):** Standard segmentation metric
*   **Pixel Accuracy:** Percentage of correctly classified pixels



In [ ]:
import torch.optim as optim
from torch.utils.data import DataLoader
from tqdm import tqdm
import numpy as np

In [ ]:
import torch
import numpy as np


def compute_iou(predictions, targets, num_classes=NUM_CLASSES):
    """Compute mean IoU across classes"""
    ious = []
    for cls in range(num_classes):
        pred_mask = (predictions == cls)
        true_mask = (targets == cls)
        intersection = (pred_mask & true_mask).sum().float()
        union = (pred_mask | true_mask).sum().float()
        iou = (intersection / union).item() if union > 0 else float('nan')
        ious.append(iou)
    valid_ious = [iou for iou in ious if not np.isnan(iou)]
    return ious, np.mean(valid_ious) if valid_ious else 0.0


def compute_pixel_accuracy(predictions, targets):
    """Compute pixel-wise accuracy"""
    correct = (predictions == targets).sum().float()
    return (correct / targets.numel()).item()


In [ ]:
import torch
import torch.nn as nn
from tqdm import tqdm

def train_one_epoch(model, dataloader, criterion, optimizer, device):
    """Train for one epoch"""
    model.train()
    total_loss, total_iou, total_acc, num_batches = 0.0, 0.0, 0.0, 0

    for images, masks, _ in tqdm(dataloader, desc="Training", leave=False):
        images = images.to(device)
        targets = masks.squeeze(1).long().to(device)

        # Forward pass
        optimizer.zero_grad()
        logits = model(images)

        # Compute loss
        loss = criterion(logits, targets)

        # Backward pass
        loss.backward()
        optimizer.step()

        # Compute metrics
        predictions = torch.argmax(logits, dim=1)
        _, iou = compute_iou(predictions, targets)
        acc = compute_pixel_accuracy(predictions, targets)

        # Accumulate
        total_loss += loss.item()
        total_iou += iou
        total_acc += acc
        num_batches += 1

    return total_loss / num_batches, total_iou / num_batches, total_acc / num_batches


def validate(model, dataloader, criterion, device):
    """Validate the model"""
    model.eval()
    total_loss, total_iou, total_acc, num_batches = 0.0, 0.0, 0.0, 0

    for images, masks, _ in tqdm(dataloader, desc="Validation", leave=False):
        images = images.to(device)
        targets = masks.squeeze(1).long().to(device)

        # Forward pass
        logits = model(images)

        # Compute loss
        loss = criterion(logits, targets)

        # Compute metrics
        predictions = torch.argmax(logits, dim=1)
        _, iou = compute_iou(predictions, targets)
        acc = compute_pixel_accuracy(predictions, targets)

        # Accumulate
        total_loss += loss.item()
        total_iou += iou
        total_acc += acc
        num_batches += 1

    return total_loss / num_batches, total_iou / num_batches, total_acc / num_batches


In [ ]:
import torch.optim as optim
from torch.utils.data import DataLoader

def train_baseline_model(
    train_dir=TRAIN_DIR, valid_dir=VALID_DIR,
    num_epochs=NUM_EPOCHS, batch_size=BATCH_SIZE, lr=LEARNING_RATE,
    subset_size=SUBSET_SIZE, save_path=None,
    unfreeze_last_n_blocks=UNFREEZE_LAST_N_BLOCKS
):
    """Train the baseline model (*all data at once, no CL*)"""
    if save_path is None:
        save_path = SAVE_DIR / "baseline_model.pth"
    
    # Datasets
    train_dataset = ContainerDefectDataset(train_dir, subset_size=subset_size)

    val_dataset = ContainerDefectDataset(
        valid_dir,
        subset_size=subset_size // 2 if subset_size else None
    )

    # Dataloaders
    train_loader = DataLoader(
        train_dataset,
        batch_size=batch_size,
        shuffle=True,
        num_workers=2,
        pin_memory=True
    )

    val_loader = DataLoader(
        val_dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=2,
        pin_memory=True
    )

    # Model
    model = ContainerDefectSegmenter(unfreeze_last_n_blocks=unfreeze_last_n_blocks).to(DEVICE)

    criterion = nn.CrossEntropyLoss()

    # Split learning rates: backbone parameters at 0.1x of head learning rate (lr)
    backbone_params = []
    head_params = []

    for name, param in model.named_parameters():
        if not param.requires_grad:
            continue
        if "backbone" in name:
            backbone_params.append(param)
        else:
            head_params.append(param)

    param_groups = []
    if backbone_params:
        param_groups.append({'params': backbone_params, 'lr': lr * 0.1})
    if head_params:
        param_groups.append({'params': head_params, 'lr': lr})

    if param_groups:
        optimizer = optim.Adam(param_groups)
    else:
        optimizer = optim.Adam(
            filter(lambda p: p.requires_grad, model.parameters()),
            lr=lr
        )
    print(f"\nBASELINE TRAINING\n")
    print(f"Device: {DEVICE}")
    print(f"Train: {len(train_dataset)} images | Val: {len(val_dataset)} images")
    print(f"Trainable params: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}\n")

    best_val_iou = 0.0
    for epoch in range(num_epochs):
        # Train
        train_loss, train_iou, train_acc = train_one_epoch(
            model, train_loader, criterion, optimizer, DEVICE
        )
        # Validate
        val_loss, val_iou, val_acc = validate(
            model, val_loader, criterion, DEVICE
        )
        print(f"Epoch [{epoch+1}/{num_epochs}]")
        print(f"  Train:  Loss={train_loss:.4f}, IoU={train_iou:.4f}, Acc={train_acc:.4f}")
        print(f"  Val:    Loss={val_loss:.4f}, IoU={val_iou:.4f}, Acc={val_acc:.4f}")

        # Save best model
        if val_iou > best_val_iou:
            best_val_iou = val_iou
            torch.save(model.state_dict(), save_path)
            print(f"Saved best model (IoU: {val_iou:.4f})")

    print("\n" + "=" * 60)
    print(f"Training complete | Best IoU: {best_val_iou:.4f}")
    print(f"Model saved: {save_path}")
    return model


In [ ]:
def load_trained_model(model_path, device='cuda'):
    """Load a saved model checkpoint"""
    model = ContainerDefectSegmenter(
        backbone_name=BACKBONE,
        num_classes=NUM_CLASSES,
        freeze_backbone=True,
        decoder_hidden_dim=DECODER_HIDDEN_DIM,
    )

    # Load weights
    model.load_state_dict(torch.load(model_path, map_location=device, weights_only=True))
    model = model.to(device)
    model.eval()

    print(f"Model loaded from: {model_path}")
    return model




```
This trains the model (no continual learning).

If complete dataset used, it serves as the upper-bound performamce for CL strategies to compare against.

Uncomment the line below to train or load.
```



In [ ]:
baseline_model = train_baseline_model()

baseline_model = load_trained_model(f"{SAVE_DIR}/baseline_model.pth")

## **SECTION 6: CONTINUAL LEARNING FRAMEWORK**

This section implements task-incremental learning where the model learns defects sequentially (Task 1 -> Task 2 -> Task 3) instead of all at once.

<br>

### CL Methods:
*   **Replay-based:** ER, ER-ACE, DER/DER++
*   **Baseline:** Naive fine-tuning (no CL)

<br>

### Evaluation Metrics:
*   **Average Accuracy:** Performance across all tasks
*   **Backward Transfer:** How much old tasks degrade
*   **Forward Transfer:** Does new learning help old tasks?




In [ ]:
from torch.utils.data import Subset
from abc import ABC, abstractmethod
import random
import copy
import json

### **Part 1: Task Splitting**

Split the dataset into sequential tasks for continual learning.

**Strategy:** Source-Incremental Split (Prefix-based)

This splitting method groups images by the container owner prefix (first character) to create a Domain/Source-Incremental challenge:

- **Task 1: Prefixes 'C'** (e.g. CSNU, CSLU)
- **Task 2: Prefixes 'T'** (e.g. TRHU, TCNU)
- **Task 3: Prefixes 'F', 'B', 'A', 'D'**
- **Task 4: Other Prefixes** (S, W, G, M, R, etc.)
- **Task 5: Mobile captures** (primarily IMG... files)

In [ ]:
import random
import torch
from torch.utils.data import Subset, DataLoader, Dataset
from typing import List, Optional



class TaskIncrementalDataset:
    """Splits dataset into sequential tasks for continual learning"""
    def __init__(
        self,
        base_dataset: ContainerDefectDataset,
        num_tasks: int = NUM_TASKS,
        mode: str = "class", # "random", "sequential", or "class"
        task_keywords: list = None,
        seed: int = 42
    ):
        self.base_dataset = base_dataset
        self.num_tasks = num_tasks
        self.seed = seed
        
        if mode == "class":
            self._split_by_class(task_keywords)
        else:
            self._split_by_index(mode == "random")

    def _split_by_index(self, shuffle: bool):
        total_size = len(self.base_dataset)
        indices = list(range(total_size))

        if shuffle:
            random.seed(self.seed)
            random.shuffle(indices)

        task_size = total_size // self.num_tasks
        self.task_indices = []

        for task_id in range(self.num_tasks):
            start_idx = task_id * task_size
            end_idx = total_size if task_id == self.num_tasks - 1 else (task_id + 1) * task_size
            self.task_indices.append(indices[start_idx:end_idx])

    def _split_by_class(self, task_keywords: list):
        """
        Splits data into tasks.
        - For OLD dataset: Splits by filename prefix (Source-Incremental).
        - For NEW dataset: Splits by COCO category IDs (Class-Incremental).
        """
        all_files = self.base_dataset.image_files
        self.task_indices = []
        
        if self.base_dataset.dataset_type == "OLD":
            self._split_by_source_old(task_keywords)
        else:
            self._split_by_category_new(task_keywords)

    def _split_by_source_old(self, task_keywords: list):
        """Original source-incremental logic for binary-mask dataset"""
        if task_keywords is None:
            task_keywords = [
                ["C"], ["T"], ["F", "B", "A", "D"], 
                ["S", "W", "G", "M", "R", "O", "U", "E", "K", "P", "H", "Y", "N", "L"], 
                ["standard_id"]
            ]
        
        self.num_tasks = len(task_keywords)
        self.task_indices = [[] for _ in range(self.num_tasks)]
        all_files = self.base_dataset.image_files
        used_indices = set()

        for t_idx, prefixes in enumerate(task_keywords):
            if "standard_id" in prefixes:
                for f_idx in range(len(all_files)):
                    if f_idx not in used_indices:
                        self.task_indices[t_idx].append(f_idx)
                        used_indices.add(f_idx)
                continue

            for f_idx, filename in enumerate(all_files):
                if f_idx in used_indices: continue
                if any(filename.upper().startswith(p.upper()) for p in prefixes):
                    self.task_indices[t_idx].append(f_idx)
                    used_indices.add(f_idx)

        print(f"\nSource-Incremental Split (OLD): {sum(len(t) for t in self.task_indices)} images")

    def _split_by_category_new(self, task_categories: list):
        """Class-incremental logic for COCO dataset"""
        if task_categories is None:
            # Grouping 6 active classes into 5 tasks
            # IDs in this dataset: 0: DEFECT, 1: PAINT, 2: PATCH, 3: REMOVE, 4: SEALANT, 5: WASH
            task_categories = [
                [1],    # Task 1: Paint
                [2],    # Task 2: Patch
                [3],    # Task 3: Remove
                [4],    # Task 4: Sealant
                [5]     # Task 5: Wash
            ]
        
        self.task_categories = task_categories # Store for label masking
        self.num_tasks = len(task_categories)
        self.task_indices = [[] for _ in range(self.num_tasks)]
        used_indices = set()
        img_to_cats = self.base_dataset.image_id_to_annotations
        
        for t_idx, cat_ids in enumerate(task_categories):
            for f_idx, img_info in enumerate(self.base_dataset.image_files):
                # 1. Handle image reuse
                if not ALLOW_REUSE and f_idx in used_indices: 
                    continue
                
                img_id = img_info['id']
                annos = img_to_cats.get(img_id, [])
                img_cats = set(a['category_id'] for a in annos)
                
                # 2. Handle composite images (multiple defect types)
                if IGNORE_COMPOSITE and len(img_cats) > 1:
                    continue
                
                # 3. Assign to task if it contains any of the target categories
                if any(c in cat_ids for c in img_cats):
                    # For safety, don't double-add the same image to the SAME task
                    if f_idx not in self.task_indices[t_idx]:
                        self.task_indices[t_idx].append(f_idx)
                        used_indices.add(f_idx)

        print(f"\nClass-Incremental Split (NEW): {sum(len(t) for t in self.task_indices)} images total")
        for i, t in enumerate(self.task_indices):
            print(f"   Task{i+1}: {len(t)} images (Categories: {task_categories[i]})")
            if len(t) == 0:
                print(f"   WARNING: Task{i+1} is EMPTY! Try increasing subset_size.")

    def get_task_dataset(self, task_id: int) -> Dataset:
        assert 0 <= task_id < self.num_tasks
        if len(self.task_indices[task_id]) == 0:
            print(f"Warning: Task {task_id+1} is empty!")
        
        subset = Subset(self.base_dataset, self.task_indices[task_id])
        
        # Apply label masking if using Class-Incremental
        if hasattr(self, 'task_categories'):
            allowed_cats = self.task_categories[task_id]
            return LabelMaskingDataset(subset, allowed_cats)
            
        return subset

    def get_task_loader(
        self,
        task_id: int,
        batch_size: int = BATCH_SIZE,
        shuffle: bool = True,
    ) -> DataLoader:
        task_dataset = self.get_task_dataset(task_id)
        return DataLoader(
            task_dataset,
            batch_size=batch_size,
            shuffle=shuffle,
            num_workers=2,
            pin_memory=True
        )

    def get_all_tasks_until(self, task_id: int) -> Subset:
        combined_indices = []
        for i in range(task_id + 1):
            combined_indices.extend(self.task_indices[i])
        return Subset(self.base_dataset, combined_indices)

class LabelMaskingDataset(Dataset):
    """
    Wrapper that masks out labels not belonging to the current task.
    Ensures 'future' classes are treated as background.
    """
    def __init__(self, dataset: Dataset, allowed_categories: List[int]):
        self.dataset = dataset
        # category_id X becomes label X in the mask. Background is 0.
        self.allowed_labels = allowed_categories + [0]
        self.allowed_labels_tensor = torch.tensor(self.allowed_labels)

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        image, mask, filename = self.dataset[idx]
        
        # Create a mask of which pixels are allowed
        is_allowed = torch.isin(mask, self.allowed_labels_tensor)
        
        # Set everything else to 0 (Background)
        masked_mask = torch.where(is_allowed, mask, torch.zeros_like(mask))
        
        return image, masked_mask, filename


### **Part 2: Replay Buffer**

**Experience Replay Buffer**

Stores samples from previous tasks to prevent catastrophic forgetting.

In [ ]:
import torch
import random


class ReplayBuffer:
    """Stores (image, mask) pairs from previous tasks"""

    def __init__(self, max_size: int = MEMORY_SIZE):
        self.max_size = max_size
        self.buffer = []
        self.task_counts = {}

    def add_samples(self, images, masks, filenames, task_id):
        batch_size = images.shape[0]

        for i in range(batch_size):
            if len(self.buffer) < self.max_size:
                self.buffer.append({
                    'image': images[i].cpu(),
                    'mask': masks[i].cpu(),
                    'filename': filenames[i],
                    'task_id': task_id
                })
                self.task_counts[task_id] = self.task_counts.get(task_id, 0) + 1
            else:
                # Buffer full, use reservoir sampling
                idx = random.randint(0, len(self.buffer) - 1)
                old_task = self.buffer[idx]['task_id']
                self.task_counts[old_task] -= 1

                self.buffer[idx] = {
                    'image': images[i].cpu(),
                    'mask': masks[i].cpu(),
                    'filename': filenames[i],
                    'task_id': task_id
                }
                self.task_counts[task_id] = self.task_counts.get(task_id, 0) + 1

    def sample(self, batch_size: int, device='cuda'):
        if len(self.buffer) == 0:
            return None, None

        sample_size = min(batch_size, len(self.buffer))
        samples = random.sample(self.buffer, sample_size)

        images = torch.stack([s['image'] for s in samples]).to(device)
        masks = torch.stack([s['mask'] for s in samples]).to(device)

        return images, masks

    def __len__(self):
        return len(self.buffer)

    def get_stats(self):
        return {
            'total': len(self.buffer),
            'max_size': self.max_size,
            'task_distribution': self.task_counts
        }


### **Part 3: CL Evaluation Metrics**

Track performance across tasks to measure forgetting and transfer.

In [ ]:
import numpy as np


class CLMetricsTracker:
    """Tracks CL metrics: Average Accuracy, Backward Transfer, Forgetting"""
    def __init__(self, num_tasks: int = NUM_TASKS):
        self.num_tasks = num_tasks
        self.accuracy_matrix = np.zeros((num_tasks, num_tasks))
        self.task_names = [f"Task {i+1}" for i in range(num_tasks)]

    def update(self, current_task: int, task_accuracies: dict):
        for task_id, acc in task_accuracies.items():
            self.accuracy_matrix[current_task][task_id] = acc

    def compute_average_accuracy(self, up_to_task: int):
        accs = []
        for task_id in range(up_to_task + 1):
            acc = self.accuracy_matrix[up_to_task][task_id]
            if acc > 0:
                accs.append(acc)
        return np.mean(accs) if accs else 0.0

    def compute_forgetting(self, up_to_task: int):
        if up_to_task == 0:
            return 0.0

        forgettings = []
        for task_id in range(up_to_task):
            max_acc = np.max(self.accuracy_matrix[:up_to_task+1, task_id])
            current_acc = self.accuracy_matrix[up_to_task][task_id]
            forgetting = max_acc - current_acc
            forgettings.append(forgetting)

        return np.mean(forgettings) if forgettings else 0.0

    def compute_backward_transfer(self):
        if self.num_tasks < 2:
            return 0.0

        bwt_scores = []
        for task_id in range(self.num_tasks - 1):
            initial_acc = self.accuracy_matrix[task_id][task_id]
            final_acc = self.accuracy_matrix[- 1][task_id]
            bwt = final_acc - initial_acc
            bwt_scores.append(bwt)

        return np.mean(bwt_scores) if bwt_scores else 0.0

    def get_summary(self):
        return {
            'Average Accuracy': self.compute_average_accuracy(self.num_tasks - 1),
            'Forgetting': self.compute_forgetting(self.num_tasks - 1),
            'Backward Transfer': self.compute_backward_transfer(),
            'Accuracy Matrix': self.accuracy_matrix.tolist()
        }

    def print_summary(self):
        print("\nCL METRICS SUMMARY")
        print("\nAccuracy Matrix (rows= after task X, cols= on task Y)")
        print("       ", end="")
        for i in range(self.num_tasks):
            print(f"T{i+1:2d}   ", end="")
        print()

        for i in range(self.num_tasks):
            print(f"After T{i+1}: ", end="")
            for j in range(self.num_tasks):
                acc = self.accuracy_matrix[i][j]
                if acc > 0:
                    print(f"{acc:.3f} ", end="")
                else:
                    print("  -   ", end="")
            print()

        print("Summary")
        print(f"  Average Accuracy:  {self.compute_average_accuracy(self.num_tasks-1):.4f}")
        print(f"  Forgetting:        {self.compute_forgetting(self.num_tasks-1):.4f}")
        print(f"  Backward Transfer: {self.compute_backward_transfer():.4f}")


### **Part 4: Base CL Strategy**

Abstract base class for all CL strategies.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from abc import ABC, abstractmethod




class BaseCLStrategy(ABC):

    def __init__(
        self,
        model: ContainerDefectSegmenter,
        device: str = DEVICE,
        lr: float = LEARNING_RATE,
    ):
        self.model = model.to(device)
        self.device = device
        self.lr = lr
        self.criterion = nn.CrossEntropyLoss()
        # Split learning rates: backbone parameters at 0.1x of head learning rate (lr)
        backbone_params = []
        head_params = []

        for name, param in self.model.named_parameters():
            if not param.requires_grad:
                continue
            if "backbone" in name:
                backbone_params.append(param)
            else:
                head_params.append(param)

        param_groups = []
        if backbone_params:
            param_groups.append({'params': backbone_params, 'lr': lr * 0.1})
        if head_params:
            param_groups.append({'params': head_params, 'lr': lr})

        if param_groups:
            self.optimizer = optim.Adam(param_groups)
        else:
            self.optimizer = optim.Adam(
                filter(lambda p: p.requires_grad, self.model.parameters()),
                lr=lr
            )

    @abstractmethod
    def train_task(
        self,
        task_loader: DataLoader,
        task_id: int,
        num_epochs: int = NUM_EPOCHS
    ):
        pass

    def before_task(self, task_id: int):
        pass

    @abstractmethod
    def after_task(self, task_loader: DataLoader, task_id: int):
        pass

    def evaluate_on_task(self, task_loader: DataLoader):
        self.model.eval()
        total_iou, total_acc, num_batches = 0.0, 0.0, 0

        with torch.no_grad():
            for images, masks, _ in task_loader:
                images = images.to(self.device)
                targets = masks.squeeze(1).long().to(self.device)

                logits = self.model(images)
                predictions = torch.argmax(logits, dim=1)

                _, iou = compute_iou(predictions, targets)
                acc = compute_pixel_accuracy(predictions, targets)

                total_iou += iou
                total_acc += acc
                num_batches += 1

        return total_iou / num_batches if num_batches > 0 else 0.0, total_acc / num_batches if num_batches > 0 else 0.0


In [ ]:
# # Test Task Splitting
# # Create task-incremental dataset
# print("\n" + "="*60)
# print("TESTING TASK SPLITTING")
# print("="*60)

# train_dataset = ContainerDefectDataset(TRAIN_DIR, subset_size=SUBSET_SIZE)
# task_split = TaskIncrementalDataset(
#     train_dataset,
#     num_tasks=NUM_TASKS,
#     shuffle_before_split=True
# )

# # Show task loaders
# print("\n✓ Task loaders created:")
# for task_id in range(NUM_TASKS):
#     loader = task_split.get_task_loader(task_id, batch_size=BATCH_SIZE)
#     print(f"  Task {task_id+1}: {len(loader)} batches")

# print("\n✓ Task splitting test complete!")

### **Part 5: CL Strategies**

Implement different continual learning approaches:


1.   **Naive Fine-tuning:** No CL, just sequential just sequential training (baseline for forgetting)
2.   **ER (Experience Replay):** Store samples, replay during training
3.   **DER++ (Dark Experience REplay):** ER + logit distillation



**Strategy 1: Naive Fine-tuning**

In [ ]:
from tqdm import tqdm
import torch




class NaiveFineTuning(BaseCLStrategy):
    """Baseline: Just train sequentially without any CL strategy"""

    def __init__(self, model, device=DEVICE, lr=1e-3):
        super().__init__(model, device, lr)
        self.name = "Naive Fine-tuning"

    def train_task(self, task_loader, task_id, num_epochs=NUM_EPOCHS):
        self.model.train()
        print(f"Training Task {task_id+1} - {self.name}")

        avg_loss = 0.0
        avg_iou = 0.0

        for epoch in range(num_epochs):
            total_loss, total_iou, num_batches = 0.0, 0.0, 0

            for images, masks, _ in tqdm(task_loader, desc=f"Epoch {epoch+1}/{num_epochs}", leave=False):
                images = images.to(self.device)
                targets = masks.squeeze(1).long().to(self.device)

                self.optimizer.zero_grad()
                logits = self.model(images)
                loss = self.criterion(logits, targets)
                loss.backward()
                self.optimizer.step()

                predictions = torch.argmax(logits, dim=1)
                _, iou = compute_iou(predictions, targets)

                total_loss += loss.item()
                total_iou += iou
                num_batches += 1

            avg_loss = total_loss / num_batches
            avg_iou = total_iou / num_batches
            print(f"  Epoch {epoch+1}: Loss={avg_loss:.4f}, IoU={avg_iou:.4f}")
        return {'loss': avg_loss, 'iou': avg_iou}

    def after_task(self, task_loader, task_id):
        pass


**Strategy 2: Experience Replay (ER)**

In [ ]:
from tqdm import tqdm
import torch





class ExperienceReplay(BaseCLStrategy):
    """Experience Replay: Store samples from old tasks and replay during training"""

    def __init__(self, model, device=DEVICE, lr=1e-3,
                 buffer_size=MEMORY_SIZE, samples_per_task=100):
        super().__init__(model, device, lr)
        self.name = "Experience Replay (ER)"
        self.buffer = ReplayBuffer(max_size=buffer_size)
        self.samples_per_task = samples_per_task

    def train_task(self, task_loader, task_id, num_epochs=NUM_EPOCHS):
        """Train on current task + replay samples"""
        self.model.train()

        print(f"Training Task {task_id+1} - {self.name}")
        print(f"Buffer: {len(self.buffer)}/{self.buffer.max_size} samples")

        avg_loss = 0.0
        avg_iou = 0.0

        for epoch in range(num_epochs):
            total_loss, total_iou, num_batches = 0.0, 0.0, 0

            for images, masks, filenames in tqdm(task_loader, desc=f"Epoch {epoch+1}/{num_epochs}", leave=False):
                images = images.to(self.device)
                targets = masks.squeeze(1).long().to(self.device)

                # Get replay samples if buffer not empty
                if len(self.buffer) > 0:
                    replay_batch_size = images.shape[0] // 2 # Half batch from buffer
                    if replay_batch_size > 0:
                        replay_images, replay_masks = self.buffer.sample(
                            batch_size=replay_batch_size,
                            device=self.device
                        )

                        if replay_images is not None:
                            images = torch.cat([images, replay_images], dim=0)
                            replay_targets = replay_masks.squeeze(1).long()
                            targets = torch.cat([targets, replay_targets], dim=0)

                self.optimizer.zero_grad()
                logits = self.model(images)
                loss = self.criterion(logits, targets)
                loss.backward()
                self.optimizer.step()

                predictions = torch.argmax(logits, dim=1)
                _, iou = compute_iou(predictions, targets)

                total_loss += loss.item()
                total_iou += iou
                num_batches += 1

            avg_loss = total_loss / num_batches
            avg_iou = total_iou / num_batches
            print(f"  Epoch {epoch+1}: Loss={avg_loss:.4f}, IoU={avg_iou:.4f}")

        return {'loss': avg_loss, 'iou': avg_iou}

    def after_task(self, task_loader, task_id):
        samples_added = 0
        for images, masks, filenames in task_loader:
            if samples_added >= self.samples_per_task:
                break

            self.buffer.add_samples(images, masks, filenames, task_id)
            samples_added += images.shape[0]


**Strategy 3: DER++ (Dark Exprience Replay++)**

In [ ]:
from tqdm import tqdm
import torch
import torch.nn as nn
import random




class DarkExperienceReplayPP(BaseCLStrategy):
    """DER++: Experience Replay + Knowledge Distillation"""

    def __init__(self, model, device=DEVICE, lr=1e-3,
                 buffer_size=MEMORY_SIZE, samples_per_task=100,
                 alpha=0.5, beta=0.5):
        super().__init__(model, device, lr)
        self.name = "DER++"
        self.buffer = []
        self.buffer_size = buffer_size
        self.samples_per_task = samples_per_task
        self.alpha = alpha
        self.beta = beta

    def train_task(self, task_loader, task_id, num_epochs=NUM_EPOCHS):
        """Train with replay + logit distillation."""
        self.model.train()

        print(f"Training Task {task_id+1} - {self.name}")
        print(f"Buffer: {len(self.buffer)}/{self.buffer_size} samples")
        print(f"{'─'*60}")

        avg_loss = 0.0
        avg_iou = 0.0

        for epoch in range(num_epochs):
            total_loss, total_iou, num_batches = 0.0, 0.0, 0

            for images, masks, filenames in tqdm(task_loader, desc=f"Epoch {epoch+1}/{num_epochs}", leave=False):
                images = images.to(self.device)
                targets = masks.squeeze(1).long().to(self.device)

                self.optimizer.zero_grad()
                logits = self.model(images)
                loss_ce = self.criterion(logits, targets)
                total_loss_value = loss_ce

                if len(self.buffer) > 0:
                    replay_batch_size = images.shape[0] // 2
                    if replay_batch_size > 0:
                        batch_size = min(replay_batch_size, len(self.buffer))
                        buffer_samples = random.sample(self.buffer, batch_size)

                        replay_images = torch.stack([s['image'] for s in buffer_samples]).to(self.device)
                        replay_masks = torch.stack([s['mask'] for s in buffer_samples]).to(self.device)
                        replay_logits = torch.stack([s['logits'] for s in buffer_samples]).to(self.device)

                        current_replay_logits = self.model(replay_images)

                        replay_targets = replay_masks.squeeze(1).long()
                        loss_replay_ce = self.criterion(current_replay_logits, replay_targets)

                        loss_distill = nn.functional.mse_loss(current_replay_logits, replay_logits)

                        total_loss_value = loss_ce + self.beta * loss_replay_ce + self.alpha * loss_distill

                total_loss_value.backward()
                self.optimizer.step()

                predictions = torch.argmax(logits, dim=1)
                _, iou = compute_iou(predictions, targets)

                total_loss += total_loss_value.item()
                total_iou += iou
                num_batches += 1

            avg_loss = total_loss / num_batches
            avg_iou = total_iou / num_batches
            print(f"  Epoch {epoch+1}: Loss={avg_loss:.4f}, IoU={avg_iou:.4f}")

        return {'loss': avg_loss, 'iou': avg_iou}

    def after_task(self, task_loader, task_id):
        self.model.eval()
        samples_added = 0

        with torch.no_grad():
            for images, masks, filenames in task_loader:
                if samples_added >= self.samples_per_task:
                    break

                images_gpu = images.to(self.device)
                logits = self.model(images_gpu)

                # Store (image, mask, logits)
                for i in range(images.shape[0]):
                    if len(self.buffer) < self.buffer_size:
                        self.buffer.append({
                            'image': images[i].cpu(),
                            'mask': masks[i].cpu(),
                            'logits': logits[i].cpu(),
                            'task_id': task_id
                        })
                    else:
                        # Reservoir sampling
                        idx = random.randint(0, len(self.buffer) - 1)
                        self.buffer[idx] = {
                            'image': images[i].cpu(),
                            'mask': masks[i].cpu(),
                            'logits': logits[i].cpu(),
                            'task_id': task_id
                        }

                    samples_added += 1
                    if samples_added >= self.samples_per_task:
                        break

        self.model.train()


**Strategy 4: Meta Experience Replay (MER)**

In [ ]:
from tqdm import tqdm
import torch





class MetaExperienceReplay(BaseCLStrategy):
    """
    MER: Meta-Experience Replay — faithful port of Mammoth's Alg 6.
    Two-level Reptile: within-batch (beta) + across-batch (gamma).
    Riemer et al., 2019.
    """

    def __init__(self, model, device=DEVICE, lr=1e-3,
                 buffer_size=MEMORY_SIZE, samples_per_task=100,
                 beta=MER_BETA,
                 gamma=MER_GAMMA,
                 batch_num=MER_BATCH_NUM):
        super().__init__(model, device, lr)
        self.name = "Meta Experience Replay (MER)"
        self.buffer = ReplayBuffer(max_size=buffer_size)
        self.samples_per_task = samples_per_task
        self.beta = beta
        self.gamma = gamma
        self.batch_num = batch_num

    def train_task(self, task_loader, task_id, num_epochs=NUM_EPOCHS):
        """Train on current task + replay using Mammoth's two-level Reptile updates"""
        self.model.train()

        print(f"Training Task {task_id+1} - {self.name}")
        print(f"Buffer: {len(self.buffer)}/{self.buffer.max_size} samples")
        print(f"beta={self.beta}, gamma={self.gamma}, batch_num={self.batch_num}")

        avg_loss = 0.0
        avg_iou = 0.0

        # Filter parameters that require gradients
        trainable_params = [p for p in self.model.parameters() if p.requires_grad]

        for epoch in range(num_epochs):
            total_loss, total_iou, num_batches = 0.0, 0.0, 0

            for images, masks, filenames in tqdm(task_loader, desc=f"Epoch {epoch+1}/{num_epochs}", leave=False):
                images = images.to(self.device)
                targets = masks.squeeze(1).long().to(self.device)

                # --- 1. Level 2 across-batch anchor (gamma) ---
                # Done parameter-wise in-place to avoid massive contiguous flat tensor allocation in GPU
                theta_A0 = [p.data.clone() for p in trainable_params]

                for _ in range(self.batch_num):
                    # --- 2. Level 1 within-batch anchor (beta) ---
                    theta_Wi0 = [p.data.clone() for p in trainable_params]

                    # Construct combined batch: concat buffer + current task data
                    if len(self.buffer) > 0:
                        replay_batch_size = max(1, images.shape[0] // 2)
                        buf_images, buf_masks = self.buffer.sample(
                            batch_size=replay_batch_size,
                            device=self.device
                        )
                        if buf_images is not None:
                            batch_images = torch.cat([buf_images, images], dim=0)
                            buf_targets = buf_masks.squeeze(1).long().to(self.device)
                            batch_targets = torch.cat([buf_targets, targets], dim=0)
                        else:
                            batch_images, batch_targets = images, targets
                    else:
                        batch_images, batch_targets = images, targets

                    # Within-batch step
                    self.optimizer.zero_grad()
                    logits = self.model(batch_images)
                    loss_val = self.criterion(logits, batch_targets)
                    loss_val.backward()
                    # Clip gradients to ensure stable weight adjustments in ViT/DINOv3 backbones
                    torch.nn.utils.clip_grad_norm_(self.model.parameters(), max_norm=1.0)
                    self.optimizer.step()

                    # Within-batch Reptile meta-update (beta)
                    for p, w_i in zip(trainable_params, theta_Wi0):
                        p.data.copy_(w_i + self.beta * (p.data - w_i))

                # --- 3. Level 2 across-batch Reptile meta-update (gamma) ---
                for p, w_a in zip(trainable_params, theta_A0):
                    p.data.copy_(w_a + self.gamma * (p.data - w_a))

                # 4. Evaluate metrics on the current batch after full meta-updates
                with torch.no_grad():
                    logits = self.model(images)
                    loss_eval = self.criterion(logits, targets)
                    predictions = torch.argmax(logits, dim=1)
                    _, iou = compute_iou(predictions, targets)

                total_loss += loss_eval.item()
                total_iou += iou
                num_batches += 1

            avg_loss = total_loss / num_batches
            avg_iou = total_iou / num_batches
            print(f"  Epoch {epoch+1}: Loss={avg_loss:.4f}, IoU={avg_iou:.4f}")

        return {'loss': avg_loss, 'iou': avg_iou}

    def after_task(self, task_loader, task_id):
        samples_added = 0
        for images, masks, filenames in task_loader:
            if samples_added >= self.samples_per_task:
                break

            self.buffer.add_samples(images, masks, filenames, task_id)
            samples_added += images.shape[0]


**Strategy 5: Synaptic Intelligence (SI)**

In [ ]:
from tqdm import tqdm
import torch
import torch.nn as nn




class SynapticIntelligence(BaseCLStrategy):
    """
    SI: Synaptic Intelligence — faithful port of Mammoth's implementation.
    Zenke et al., 2017 — "Continual Learning Through Synaptic Intelligence"

    Regularization-based CL: tracks per-parameter importance (big_omega)
    accumulated over training trajectory, penalises changes to important
    parameters when learning new tasks. No replay buffer required.
    """

    def __init__(self, model, device=DEVICE, lr=1e-3,
                 c=SI_C, xi=SI_XI):
        super().__init__(model, device, lr)
        self.name = "Synaptic Intelligence (SI)"
        self.c = c      # surrogate loss weight — scales penalty strength
        self.xi = xi    # damping term — prevents division by zero in omega update

        # Get list of parameters requiring gradients
        self.trainable_params = [p for p in self.model.parameters() if p.requires_grad]

        # Parameter-wise buffers (OOM-proof, no massive flat tensors created in training loop)
        self.checkpoint = {p: p.data.clone() for p in self.trainable_params}
        self.big_omega = {}     # accumulated importance weights (Ω)
        self.small_omega = {p: torch.zeros_like(p.data) for p in self.trainable_params}
        self.has_big_omega = False

    def train_task(self, task_loader, task_id, num_epochs=NUM_EPOCHS):
        """Train task while tracking parameter contributions and penalizing changes to important weights"""
        self.model.train()

        print(f"Training Task {task_id+1} - {self.name}")
        print(f"c={self.c}, xi={self.xi}")
        print(f"{'─'*60}")

        avg_loss = 0.0
        avg_iou = 0.0

        for epoch in range(num_epochs):
            total_loss, total_iou, num_batches = 0.0, 0.0, 0

            for images, masks, filenames in tqdm(task_loader, desc=f"Epoch {epoch+1}/{num_epochs}", leave=False):
                images = images.to(self.device)
                targets = masks.squeeze(1).long().to(self.device)

                # Snapshot parameter data before step (needed for small_omega update)
                pre_params = {p: p.data.clone() for p in self.trainable_params}

                self.optimizer.zero_grad()
                logits = self.model(images)
                loss = self.criterion(logits, targets)
                loss.backward()

                # Capture raw task gradients immediately after backward, before penalty injection or optimizer step
                raw_grads = {p: p.grad.data.clone() for p in self.trainable_params if p.grad is not None}

                # Inject penalty gradients into existing grads (Tasks 2+)
                if self.has_big_omega:
                    for p in self.trainable_params:
                        if p.grad is not None:
                            # penalty grad = c * 2 * big_omega * (theta - theta*)
                            penalty_g = self.c * 2 * self.big_omega[p] * (p.data - self.checkpoint[p])
                            p.grad.data.add_(penalty_g)

                # Gradient value clipping (Mammoth uses clip_grad_value_=1.0)
                torch.nn.utils.clip_grad_value_(self.trainable_params, 1.0)
                self.optimizer.step()

                # Accumulate small_omega: ω += g * Δθ (gradient dotted with parameter displacement)
                # Done parameter-wise to avoid flattening overhead, using raw captured gradients
                for p in self.trainable_params:
                    if p in raw_grads:
                        delta_theta = pre_params[p] - p.data
                        self.small_omega[p] += raw_grads[p] * delta_theta

                # Metrics
                with torch.no_grad():
                    preds = torch.argmax(logits, dim=1)
                    _, iou = compute_iou(preds, targets)

                total_loss += loss.item()
                total_iou += iou
                num_batches += 1

            avg_loss = total_loss / num_batches
            avg_iou = total_iou / num_batches
            print(f"  Epoch {epoch+1}: Loss={avg_loss:.4f}, IoU={avg_iou:.4f}")

        return {'loss': avg_loss, 'iou': avg_iou}

    def after_task(self, task_loader, task_id):
        """
        End-of-task Ω update (Mammoth's end_task):
        Ω += ω / ((θ - θ*)² + ξ)
        Then reset θ* and ω for the next task.
        """
        if not self.has_big_omega:
            self.big_omega = {p: torch.zeros_like(p.data) for p in self.trainable_params}
            self.has_big_omega = True

        for p in self.trainable_params:
            # Ω += ω / ((θ - θ*)² + ξ)
            displacement = (p.data - self.checkpoint[p]) ** 2
            self.big_omega[p] += self.small_omega[p] / (displacement + self.xi)

            # Reset checkpoints and small omegas
            self.checkpoint[p].copy_(p.data)
            self.small_omega[p].zero_()


**Strategy 6: Learning Without Forgetting (LwF)**

In [ ]:
import copy
from tqdm import tqdm
import torch
import torch.nn.functional as F




class LearningWithoutForgetting(BaseCLStrategy):
    """
    LwF: Learning without Forgetting — adapted from Mammoth for segmentation.
    Li & Hoiem, 2017.

    Distills knowledge from the frozen old model's pixel-wise logits
    instead of replaying stored samples. No buffer required.
    """

    def __init__(self, model, device=DEVICE, lr=1e-3,
                 alpha=LWF_ALPHA, temperature=LWF_TEMPERATURE):
        super().__init__(model, device, lr)
        self.name = "Learning Without Forgetting (LwF)"
        self.alpha = alpha
        self.temperature = temperature
        self.old_model = None

    def _distillation_loss(self, new_logits, old_logits):
        """
        Pixel-wise KL divergence between old and new model outputs.
        Operates on (B, C, H, W) segmentation logits.
        T² scaling follows Hinton et al. knowledge distillation convention.
        """
        assert new_logits.shape == old_logits.shape, (
            f"Shape mismatch: new={new_logits.shape}, old={old_logits.shape}"
        )
        T = self.temperature
        old_soft = F.softmax(old_logits / T, dim=1)
        new_logsoft = F.log_softmax(new_logits / T, dim=1)
        # Compute KL divergence per pixel (sum over channels), then average over batch and space
        kl_pixel = F.kl_div(new_logsoft, old_soft, reduction='none').sum(dim=1)
        loss = kl_pixel.mean() * (T ** 2)
        return loss

    def before_task(self, task_id):
        """
        Snapshot the current model as the frozen teacher before training
        the new task. Only fires from Task 2 onward.
        """
        if task_id > 0:
            self.old_model = copy.deepcopy(self.model)
            self.old_model.eval()
            for p in self.old_model.parameters():
                p.requires_grad = False

    def train_task(self, task_loader, task_id, num_epochs=NUM_EPOCHS):
        self.model.train()

        print(f"Training Task {task_id+1} - {self.name}")
        print(f"alpha={self.alpha}, temperature={self.temperature}")
        print(f"Distillation: {'ON' if self.old_model is not None else 'OFF (Task 1)'}")
        print(f"{'─'*60}")

        avg_loss = 0.0
        avg_iou = 0.0

        for epoch in range(num_epochs):
            total_loss, total_iou, num_batches = 0.0, 0.0, 0

            for images, masks, filenames in tqdm(task_loader, desc=f"Epoch {epoch+1}/{num_epochs}", leave=False):
                images = images.to(self.device)
                targets = masks.squeeze(1).long().to(self.device)

                self.optimizer.zero_grad()
                new_logits = self.model(images)

                loss = self.criterion(new_logits, targets)

                if self.old_model is not None:
                    with torch.no_grad():
                        old_logits = self.old_model(images)
                    loss += self.alpha * self._distillation_loss(new_logits, old_logits)

                loss.backward()
                torch.nn.utils.clip_grad_norm_(self.model.parameters(), max_norm=1.0)
                self.optimizer.step()

                with torch.no_grad():
                    preds = torch.argmax(new_logits, dim=1)
                    _, iou = compute_iou(preds, targets)

                total_loss += loss.item()
                total_iou += iou
                num_batches += 1

            avg_loss = total_loss / num_batches
            avg_iou = total_iou / num_batches
            print(f"  Epoch {epoch+1}: Loss={avg_loss:.4f}, IoU={avg_iou:.4f}")

        return {'loss': avg_loss, 'iou': avg_iou}

    def after_task(self, task_loader, task_id):
        pass


**Strategy 7: Slow Learner Classifier Alignment (SLCA)**

In [ ]:
import copy
from tqdm import tqdm
import torch
import torch.nn.functional as F




class SlowLearnerClassifierAlignment(BaseCLStrategy):
    """
    SLCA: Slow Learner with Classifier Alignment — adapted from Mammoth for segmentation.
    Zhang et al., 2023.

    Since the backbone is frozen, we only perform the Classifier Alignment (CA) phase
    after each task. Synthetic prototypes are generated from the frozen backbone
    features and used to re-align the decoder head in a balanced multi-class batch
    to counter recency bias.
    """

    def __init__(self, model, device=DEVICE, lr=1e-3,
                 ca_epochs=SLCA_CA_EPOCHS,
                 logit_norm=SLCA_LOGIT_NORM):
        super().__init__(model, device, lr)
        self.name = "SLCA"
        self.ca_epochs = ca_epochs
        self.logit_norm = logit_norm

        self.class_means = {}
        self.seen_tasks = []

    def _extract_backbone_features(self, images):
        """Extract spatial features from frozen backbone."""
        with torch.no_grad():
            features = self.model.backbone(images)
        return features

    def _compute_class_prototypes(self, task_loader, task_classes):
        """
        Compute mean feature prototype per class from the frozen backbone.
        Operates pixel-wise on spatial feature maps.
        """
        self.model.eval()
        feature_sums = {c: None for c in task_classes}
        feature_counts = {c: 0 for c in task_classes}

        with torch.no_grad():
            for images, masks, _ in tqdm(task_loader, desc="Computing prototypes", leave=False):
                images = images.to(self.device)
                masks = masks.squeeze(1).long().to(self.device)

                features = self._extract_backbone_features(images)
                B, C, H_f, W_f = features.shape

                masks_resized = F.interpolate(
                    masks.float().unsqueeze(1), size=(H_f, W_f),
                    mode='nearest'
                ).squeeze(1).long()

                for c in task_classes:
                    pixel_mask = (masks_resized == c)
                    if pixel_mask.sum() == 0:
                        continue
                    
                    pixel_feats = features.permute(0, 2, 3, 1)[pixel_mask]
                    if feature_sums[c] is None:
                        feature_sums[c] = pixel_feats.sum(dim=0)
                    else:
                        feature_sums[c] += pixel_feats.sum(dim=0)
                    feature_counts[c] += pixel_mask.sum().item()

        prototypes = {}
        for c in task_classes:
            if feature_counts[c] > 0:
                prototypes[c] = (feature_sums[c] / feature_counts[c]).cpu()

        self.model.train()
        return prototypes

    def _classifier_alignment(self, task_id):
        """
        CA phase: re-balance the segmentation head using stored class prototypes.
        Updates the decoder head using a single multi-class batch containing all
        seen class prototypes expanded to correct spatial dimensions.
        """
        if len(self.class_means) < 2:
            return

        print(f"  Running Classifier Alignment over {len(self.class_means)} classes...")

        ca_optimizer = torch.optim.Adam(
            [p for p in self.model.head.parameters() if p.requires_grad],
            lr=self.optimizer.param_groups[0]['lr'] * 0.1
        )

        class_ids = sorted(self.class_means.keys())
        
        # Stack prototypes and expand to match target feature map shape (N_seen, C, H_f, W_f)
        protos_stacked = torch.stack([self.class_means[c] for c in class_ids]).to(self.device)
        H_f = W_f = IMAGE_SIZE // PATCH_SIZE
        proto_spatial = protos_stacked.unsqueeze(-1).unsqueeze(-1).expand(-1, -1, H_f, W_f)

        for ca_epoch in range(self.ca_epochs):
            ca_optimizer.zero_grad()
            
            # Forward directly through the decoder head
            logits = self.model.head(proto_spatial)

            if self.logit_norm > 0:
                norm = torch.norm(logits, p=2, dim=1, keepdim=True).clamp(min=1e-8)
                logits = logits / norm * self.logit_norm

            H_out, W_out = logits.shape[2], logits.shape[3]
            targets = torch.tensor(class_ids, dtype=torch.long, device=self.device)
            targets = targets.view(-1, 1, 1).expand(-1, H_out, W_out)

            loss = self.criterion(logits, targets)
            loss.backward()
            ca_optimizer.step()

            print(f"    CA Epoch {ca_epoch+1}/{self.ca_epochs}: Loss={loss.item():.4f}")

    def train_task(self, task_loader, task_id, num_epochs=NUM_EPOCHS):
        self.model.train()

        print(f"Training Task {task_id+1} - {self.name}")
        print(f"ca_epochs={self.ca_epochs}, logit_norm={self.logit_norm}")
        print(f"{'─'*60}")

        avg_loss = 0.0
        avg_iou = 0.0

        for epoch in range(num_epochs):
            total_loss, total_iou, num_batches = 0.0, 0.0, 0

            for images, masks, filenames in tqdm(task_loader, desc=f"Epoch {epoch+1}/{num_epochs}", leave=False):
                images = images.to(self.device)
                targets = masks.squeeze(1).long().to(self.device)

                self.optimizer.zero_grad()
                logits = self.model(images)
                loss = self.criterion(logits, targets)
                loss.backward()
                torch.nn.utils.clip_grad_norm_(self.model.parameters(), max_norm=1.0)
                self.optimizer.step()

                with torch.no_grad():
                    preds = torch.argmax(logits, dim=1)
                    _, iou = compute_iou(preds, targets)

                total_loss += loss.item()
                total_iou += iou
                num_batches += 1

            avg_loss = total_loss / num_batches
            avg_iou = total_iou / num_batches
            print(f"  Epoch {epoch+1}: Loss={avg_loss:.4f}, IoU={avg_iou:.4f}")

        return {'loss': avg_loss, 'iou': avg_iou}

    def after_task(self, task_loader, task_id):
        """
        Extract class prototypes for background (0) and new classes,
        then run Decoder/Classifier Alignment.
        """
        task_classes = [0, task_id + 1]

        print(f"\nAfter Task {task_id+1}: computing prototypes for classes {task_classes}")
        new_prototypes = self._compute_class_prototypes(task_loader, task_classes)
        self.class_means.update(new_prototypes)
        self.seen_tasks.append(task_id)

        if task_id > 0:
            self._classifier_alignment(task_id)


**Strategy 8: Experience Replay ACE (ER-ACE)**

In [ ]:
from tqdm import tqdm
import torch





class ExperienceReplayACE(BaseCLStrategy):
    """
    ER-ACE: Experience Replay with Asymmetric Cross-Entropy.
    Adapted from Mammoth for dense segmentation (B, C, H, W) logits.
    Buzzega et al., 2022.

    Asymmetric loss treatment:
    - Current task samples: CE masked to only seen classes (prevents
      the model from being penalised for not predicting unseen classes)
    - Buffer samples: standard CE (classes were seen during their task)

    Directly addresses background drift in class-incremental segmentation:
    old classes in buffer samples are no longer incorrectly penalised
    as background when new classes are introduced.
    """

    def __init__(self, model, device=DEVICE, lr=1e-3,
                 buffer_size=MEMORY_SIZE, samples_per_task=100):
        super().__init__(model, device, lr)
        self.name = "ER-ACE"
        self.buffer = ReplayBuffer(max_size=buffer_size)
        self.samples_per_task = samples_per_task
        self.seen_classes = set()

    def _ace_loss(self, logits, targets, present_classes):
        """
        Asymmetric CE loss for current task samples.
        Masks out logits for classes not yet seen, preventing the model
        from being penalised for classes it hasn't been trained on yet.
        """
        allowed = self.seen_classes.union(present_classes).union({0})

        mask = torch.zeros(logits.shape[1], device=logits.device)
        for c in allowed:
            if c < logits.shape[1]:
                mask[c] = 1.0

        mask = mask.view(1, -1, 1, 1)
        masked_logits = logits.masked_fill(mask == 0, float('-inf'))

        return self.criterion(masked_logits, targets)

    def train_task(self, task_loader, task_id, num_epochs=NUM_EPOCHS):
        self.model.train()

        print(f"Training Task {task_id+1} - {self.name}")
        print(f"Buffer: {len(self.buffer)}/{self.buffer.max_size} samples")
        print(f"Seen classes so far: {sorted(self.seen_classes)}")
        print(f"{'─'*60}")

        avg_loss = 0.0
        avg_iou = 0.0

        for epoch in range(num_epochs):
            total_loss, total_iou, num_batches = 0.0, 0.0, 0

            for images, masks, filenames in tqdm(task_loader, desc=f"Epoch {epoch+1}/{num_epochs}", leave=False):
                images = images.to(self.device)
                targets = masks.squeeze(1).long().to(self.device)

                present_classes = set(targets.unique().cpu().tolist()) - {0}
                self.seen_classes.update(present_classes)

                self.optimizer.zero_grad()
                logits = self.model(images)

                if task_id > 0:
                    loss = self._ace_loss(logits, targets, present_classes)
                else:
                    loss = self.criterion(logits, targets)

                if len(self.buffer) > 0:
                    replay_batch_size = max(1, images.shape[0] // 2)
                    buf_images, buf_masks = self.buffer.sample(
                        batch_size=replay_batch_size, device=self.device
                    )
                    if buf_images is not None:
                        buf_targets = buf_masks.squeeze(1).long()
                        buf_logits = self.model(buf_images)
                        loss_replay = self.criterion(buf_logits, buf_targets)
                        loss = loss + loss_replay

                loss.backward()
                torch.nn.utils.clip_grad_norm_(self.model.parameters(), max_norm=1.0)
                self.optimizer.step()

                with torch.no_grad():
                    preds = torch.argmax(logits, dim=1)
                    _, iou = compute_iou(preds, targets)

                total_loss += loss.item()
                total_iou += iou
                num_batches += 1

            avg_loss = total_loss / num_batches
            avg_iou = total_iou / num_batches
            print(f"  Epoch {epoch+1}: Loss={avg_loss:.4f}, IoU={avg_iou:.4f}")

        return {'loss': avg_loss, 'iou': avg_iou}

    def after_task(self, task_loader, task_id):
        """Fill buffer with reservoir sampling after each task."""
        samples_added = 0
        for images, masks, filenames in task_loader:
            if samples_added >= self.samples_per_task:
                break
            self.buffer.add_samples(images, masks, filenames, task_id)
            samples_added += images.shape[0]


**Strategy Factory**

In [ ]:
def get_cl_strategy(strategy_name, model, device=DEVICE, lr=LEARNING_RATE):
    """Factory function to create CL strategies"""
    strategies = {
        'naive': NaiveFineTuning,
        'er': ExperienceReplay,
        'der++': DarkExperienceReplayPP,
        'mer': MetaExperienceReplay,
        'si': SynapticIntelligence,
        'lwf': LearningWithoutForgetting,
        'slca': SlowLearnerClassifierAlignment,
        'er_ace': ExperienceReplayACE,
    }

    if strategy_name.lower() not in strategies:
        raise ValueError(f"Unknown strategy: {strategy_name}")

    return strategies[strategy_name.lower()](model, device, lr)


### **Part 6: CL Training Loop**





**Continual Learning Trainer**


```
Main exectution eengine that:
  1. Trains model sequentially on Task 1 -> Task 2 -> Task 3  
  2. Evaluates on ALL previous tasks after each new task
  3. Tracks metrics (accuracy matrix, forgetting)
  4. Saves checkpoints
```

In [ ]:
import torch
import json





class CLTrainer:
    """Orchestrates continual learning training and evaluation"""
    def __init__(
        self,
        strategy: BaseCLStrategy,
        task_manager: TaskIncrementalDataset,
        val_task_manager: TaskIncrementalDataset = None,
        device: str = DEVICE
    ):
        self.strategy = strategy
        self.task_manager = task_manager
        self.val_task_manager = val_task_manager
        self.device = device
        self.num_tasks = task_manager.num_tasks
        self.metrics = CLMetricsTracker(num_tasks=self.num_tasks)
        self.pixel_metrics = CLMetricsTracker(num_tasks=self.num_tasks)
        self.training_history = []

    def train(self, num_epochs_per_task: int = NUM_EPOCHS, save_dir = SAVE_DIR):
        from pathlib import Path
        save_dir = Path(save_dir)

        print(f"\nCONTINUAL LEARNING TRAINING: {self.strategy.name}\n")
        print(f"Tasks: {self.num_tasks}")
        print(f"Epochs per task: {num_epochs_per_task}")
        print(f"Device: {self.device}\n")

        for task_id in range(self.num_tasks):
            print(f"\n{'█'*60}")
            print(f"TASK {task_id + 1}/{self.num_tasks}")
            print(f"{'█'*60}")

            task_loader = self.task_manager.get_task_loader(task_id, batch_size=BATCH_SIZE, shuffle=True)
            self.strategy.before_task(task_id)
            train_metrics = self.strategy.train_task(task_loader, task_id, num_epochs=num_epochs_per_task)
            self.strategy.after_task(task_loader, task_id)

            print(f"\n  Evaluating on all tasks...")
            task_ious, task_accs = self._evaluate_all_tasks(task_id)
            self.metrics.update(task_id, task_ious)
            self.pixel_metrics.update(task_id, task_accs)
            self._print_task_results(task_id, task_ious, task_accs)

            # Save checkpoint (save only trainable parameters to save massive disk space)
            checkpoint_path = save_dir / f"{self.strategy.name.replace(' ', '_')}_task{task_id+1}.pth"
            trainable_keys = {n for n, p in self.strategy.model.named_parameters() if p.requires_grad}
            trainable_state_dict = {
                k: v for k, v in self.strategy.model.state_dict().items()
                if k in trainable_keys or "head" in k
            }
            torch.save(trainable_state_dict, checkpoint_path)
            print(f"  Checkpoint saved: {checkpoint_path}")

            self.training_history.append({
                'task_id': task_id,
                'train_metrics': train_metrics,
                'task_accuracies': task_ious,
                'task_pixel_accuracies': task_accs,
                'avg_accuracy': self.metrics.compute_average_accuracy(task_id),
                'avg_pixel_accuracy': self.pixel_metrics.compute_average_accuracy(task_id),
                'forgetting': self.metrics.compute_forgetting(task_id),
                'pixel_forgetting': self.pixel_metrics.compute_forgetting(task_id)
            })

        print("TRAINING COMPLETE")
        print("\n" + "="*60 + "\nSUMMARY OF IoU METRICS\n" + "="*60)
        self.metrics.print_summary()
        
        print("\n" + "="*60 + "\nSUMMARY OF PIXEL ACCURACY METRICS\n" + "="*60)
        self.pixel_metrics.print_summary()

        return {
            'metrics': self.metrics.get_summary(),
            'pixel_metrics': self.pixel_metrics.get_summary(),
            'history': self.training_history,
            'strategy_name': self.strategy.name
        }

    def _evaluate_all_tasks(self, current_task: int):
        task_ious = {}
        task_accs = {}

        for eval_task_id in range(current_task + 1):
            if self.val_task_manager is not None:
                eval_loader = self.val_task_manager.get_task_loader(eval_task_id, batch_size=BATCH_SIZE, shuffle=False)
            else:
                eval_loader = self.task_manager.get_task_loader(eval_task_id, batch_size=BATCH_SIZE, shuffle=False)

            iou, acc = self.strategy.evaluate_on_task(eval_loader)
            task_ious[eval_task_id] = iou
            task_accs[eval_task_id] = acc

        return task_ious, task_accs

    def _print_task_results(self, current_task: int, task_ious: dict, task_accs: dict):
        print(f"\n  Performance after Task {current_task + 1}:")
        for task_id in task_ious.keys():
            iou = task_ious[task_id]
            acc = task_accs[task_id]
            print(f"    Task {task_id + 1}: IoU = {iou:.4f} | Pixel Acc = {acc:.4f}")

        avg_iou = self.metrics.compute_average_accuracy(current_task)
        forgetting_iou = self.metrics.compute_forgetting(current_task)

        avg_acc = self.pixel_metrics.compute_average_accuracy(current_task)
        forgetting_acc = self.pixel_metrics.compute_forgetting(current_task)

        print(f"  ─────────────────────────────")
        print(f"    Avg IoU:      {avg_iou:.4f} | Avg Pixel Acc: {avg_acc:.4f}")
        print(f"    Forgetting:   {forgetting_iou:.4f} | Forgetting Acc: {forgetting_acc:.4f}")

    def save_results(self, filepath: str):
        results = {
            'strategy': self.strategy.name,
            'num_tasks': self.num_tasks,
            'metrics': self.metrics.get_summary(),
            'pixel_metrics': self.pixel_metrics.get_summary(),
            'history': self.training_history
        }

        with open(filepath, 'w') as f:
            json.dump(results, f, indent=2)

        print(f"Results saved to: {filepath}")


**Convenience Function: Run Single CL Experiment**

In [ ]:
def run_cl_experiment(
    strategy_name: str,
    num_epochs_per_task: int = NUM_EPOCHS,
    subset_size: int = SUBSET_SIZE,
    save_results: bool = True,
    save_dir = SAVE_DIR,
    results_dir = RESULTS_DIR,
    unfreeze_last_n_blocks: int = UNFREEZE_LAST_N_BLOCKS
):
    from pathlib import Path
    save_dir = Path(save_dir)
    results_dir = Path(results_dir)
    """Run a complete CL experiment with a single strategy"""

    # Create datasets and task splits
    train_dataset = ContainerDefectDataset(TRAIN_DIR, subset_size=subset_size)
    val_dataset = ContainerDefectDataset(
        VALID_DIR,
        subset_size=subset_size // 2 if subset_size else None
    )

    train_tasks = TaskIncrementalDataset(train_dataset, num_tasks=NUM_TASKS, mode="class")
    val_tasks = TaskIncrementalDataset(val_dataset, num_tasks=NUM_TASKS, mode="class")

    # Create fresh model for this experiment
    model = ContainerDefectSegmenter(
        backbone_name=BACKBONE,
        num_classes=NUM_CLASSES,
        freeze_backbone=True,
        unfreeze_last_n_blocks=unfreeze_last_n_blocks,
        decoder_hidden_dim=DECODER_HIDDEN_DIM
    ).to(DEVICE)

    # Create CL strategy
    strategy = get_cl_strategy(strategy_name, model, DEVICE, lr=LEARNING_RATE)

    # Create trainer
    trainer = CLTrainer(
        strategy=strategy,
        task_manager=train_tasks,
        val_task_manager=val_tasks,
        device=DEVICE
    )

    # Run training
    results = trainer.train(num_epochs_per_task=num_epochs_per_task, save_dir=save_dir)

    # Save results
    if save_results:
        results_dir.mkdir(parents=True, exist_ok=True)
        results_file = results_dir / f"cl_results_{strategy_name}.json"
        trainer.save_results(results_file)

    return results


**Run All Strategies**

In [ ]:
def run_all_cl_strategies(
    num_epochs_per_task: int = NUM_EPOCHS,
    subset_size: int = SUBSET_SIZE,
    save_dir = SAVE_DIR,
    results_dir = RESULTS_DIR,
    strategies: list = None,
    unfreeze_last_n_blocks: int = UNFREEZE_LAST_N_BLOCKS
):
    from pathlib import Path
    save_dir = Path(save_dir)
    results_dir = Path(results_dir)
    """Run all CL strategies and compare results"""
    if strategies is None:
        strategies = ['naive', 'er', 'der++', 'mer', 'si', 'lwf', 'slca', 'er_ace']
    all_results = {}

    print("\nRUNNING ALL CL STRATEGIES\n")
    print(f"Strategies to test: {strategies}")
    print(f"Epochs per task: {num_epochs_per_task}")
    print(f"Dataset size: {subset_size if subset_size else 'FULL'}")
    print(f"Unfreeze last N blocks: {unfreeze_last_n_blocks}")

    for strategy_name in strategies:
        try:
            results = run_cl_experiment(
                strategy_name=strategy_name,
                num_epochs_per_task=num_epochs_per_task,
                subset_size=subset_size,
                save_results=True,
                save_dir=save_dir,
                results_dir=results_dir,
                unfreeze_last_n_blocks=unfreeze_last_n_blocks
            )
            all_results[strategy_name] = results
        except Exception as e:
            print(f"\nError with {strategy_name}: {e}")
            import traceback
            traceback.print_exc()
            all_results[strategy_name] = None

    _print_comparison_summary(all_results)
    generate_cl_report(all_results, save_dir=results_dir)
    return all_results


def _print_comparison_summary(all_results: dict):
    print("\nCOMPARISON SUMMARY\n")
    print(f"{'Strategy':<20} {'Avg Acc':>10} {'Forgetting':>12} {'BWT':>10}")
    print("-" * 60)

    for strategy_name, results in all_results.items():
        if results is not None:
            metrics = results['metrics']
            avg_acc = metrics['Average Accuracy']
            forgetting = metrics['Forgetting']
            bwt = metrics['Backward Transfer']
            print(f"{strategy_name.upper():<20} {avg_acc:>10.4f} {forgetting:>12.4f} {bwt:>10.4f}")
        else:
            print(f"{strategy_name.upper():<20} {'ERROR':>10} {'ERROR':>12} {'ERROR':>10}")


In [ ]:
# Test with Naive Fine-tuning (shows catastrophic forgetting)
# results_naive = run_cl_experiment('naive', num_epochs_per_task=2, subset_size=200)

# Test with Experience Replay
# results_er = run_cl_experiment('er', num_epochs_per_task=2, subset_size=200)

# Test with DER++
# results_derpp = run_cl_experiment('der++', num_epochs_per_task=2, subset_size=200)

# Run all strategies and compare
all_results = run_all_cl_strategies(num_epochs_per_task=2, subset_size=50)

###**Part 7: Visualization & Analysis**

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from typing import Dict

In [ ]:
def load_cl_results(strategy_name: str, results_dir: str = SAVE_DIR) -> dict:
  """Load saved CL experiment results"""
  filepath = f"{results_dir}/cl_results_{strategy_name}.json"

  try:
    with open(filepath, 'r') as f:
      results = json.load(f)
    return results
  except FileNotFoundError:
    print(f"Results file not found: {filepath}")
    return None

**Accuracy Matrix Heatmap**

In [ ]:
def plot_accuracy_matrix(results: dict, save_path: str = None):
    """Plot accuracy matrix as heatmap"""
    matrix = np.array(results['metrics']['Accuracy Matrix'])
    strategy_name = results.get('strategy_name', 'CL Strategy')
    mask = np.triu(np.ones_like(matrix, dtype=bool), k=1)

    plt.figure(figsize=(8, 6))
    sns.heatmap(
        matrix,
        mask=mask,
        annot=True,
        fmt='.3f',
        cmap='RdYlGn',
        vmin=0,
        vmax=1,
        cbar_kws={'label': 'IoU'},
        square=True
    )

    plt.title(f'Accuracy Matrix - {strategy_name}', fontsize=14, fontweight='bold')
    plt.xlabel('Task Evaluated On', fontsize=12)
    plt.ylabel('After Training Task', fontsize=12)

    # Set tick labels
    num_tasks = matrix.shape[0]
    tick_labels = [f'T{i+1}' for i in range(num_tasks)]
    plt.xticks(np.arange(num_tasks) + 0.5, tick_labels)
    plt.yticks(np.arange(num_tasks) + 0.5, tick_labels, rotation=0)

    plt.tight_layout()

    if save_path:
        plt.savefig(save_path, dpi=300, bbox_inches='tight')


**Task Performance Over Time**

In [ ]:
def plot_task_performance_curves(results: dict, save_path: str = None):
    """Plot how performance on each task changes as new tasks are learned"""
    matrix = np.array(results['metrics']['Accuracy Matrix'])
    strategy_name = results.get('strategy_name', 'CL Strategy')
    num_tasks = matrix.shape[0]

    plt.figure(figsize=(10, 6))

    for task_id in range(num_tasks):
        performance = []
        x_points = []

        for train_task in range(task_id, num_tasks):
            if matrix[train_task, task_id] > 0:
                performance.append(matrix[train_task, task_id])
                x_points.append(train_task + 1)

        plt.plot(
            x_points,
            performance,
            marker='o',
            linewidth=2,
            markersize=8,
            label=f'Task {task_id + 1}'
        )

    plt.xlabel('After Training Task', fontsize=12)
    plt.ylabel('IoU', fontsize=12)
    plt.title(f'Task Performance Over Time - {strategy_name}', fontsize=14, fontweight='bold')
    plt.legend(loc='best')
    plt.grid(True, alpha=0.3)
    plt.xticks(range(1, num_tasks + 1))
    plt.ylim([0, 1])
    plt.tight_layout()

    if save_path:
        plt.savefig(save_path, dpi=300, bbox_inches='tight')


**Compare Multiple Strategies**

In [ ]:
def clean_strategy_name(key: str) -> str:
    import re
    # Convert to lowercase
    k = key.lower()
    # Strip any trailing _run_X or _runX or _run or _run_run_X
    k = re.sub(r'_(run_\d+|run\d+|run)$', '', k)
    k = re.sub(r'_(run)$', '', k)
    return k


def get_run_type(run_name: str, project_root: Path) -> str:
    models_dir = project_root / "models" / run_name
    if not models_dir.exists():
        try:
            run_num = int(run_name.replace("run_", ""))
            return "Unfrozen" if run_num >= 7 else "Frozen"
        except ValueError:
            return "Frozen"
            
    pth_files = list(models_dir.glob("*.pth"))
    if not pth_files:
        return "Frozen"
        
    try:
        import torch
        state_dict = torch.load(pth_files[0], map_location="cpu")
        has_backbone = any("backbone" in k for k in state_dict.keys())
        return "Unfrozen" if has_backbone else "Frozen"
    except Exception:
        size_mb = pth_files[0].stat().st_size / (1024 * 1024)
        return "Unfrozen" if size_mb > 50.0 else "Frozen"


def plot_combined_comparison(strategy_results: dict, save_path: Path, project_root: Path):
    """Plot accuracy, forgetting, and backward transfer comparison side-by-side"""
    import numpy as np
    
    # 1. Group the results by strategy name and backbone type
    grouped_results = {}
    has_frozen = False
    has_unfrozen = False
    
    for key, data in strategy_results.items():
        if data is None or 'metrics' not in data:
            continue
            
        # Extract clean strategy name
        strat_name = clean_strategy_name(key)
        run_name = data.get('run_dir', 'Direct')
        
        # Determine run type
        rtype = get_run_type(run_name, project_root)
        if rtype == "Frozen":
            has_frozen = True
        else:
            has_unfrozen = True
            
        if strat_name not in grouped_results:
            grouped_results[strat_name] = {}
        grouped_results[strat_name][rtype] = data

    # 2. Check if we should do side-by-side (both frozen and unfrozen are present)
    use_side_by_side = has_frozen and has_unfrozen

    if not grouped_results:
        print("No strategy results found. Comparison plot aborted.")
        return

    strategy_display_names = {
        'naive': 'Naive',
        'er': 'ER',
        'der++': 'DER++',
        'mer': 'MER',
        'si': 'SI',
        'lwf': 'LwF',
        'slca': 'SLCA',
        'er_ace': 'ER-ACE'
    }

    fig, axes = plt.subplots(1, 3, figsize=(16, 6))

    if use_side_by_side:
        # Grouped side-by-side plotting
        strategies = sorted(list(grouped_results.keys()))
        labels = [strategy_display_names.get(s, s.upper()) for s in strategies]
        num_strategies = len(strategies)
        x = np.arange(num_strategies)
        width = 0.35

        frozen_accs, unfrozen_accs = [], []
        frozen_forget, unfrozen_forget = [], []
        frozen_bwt, unfrozen_bwt = [], []

        for s in strategies:
            # Frozen
            f_data = grouped_results[s].get("Frozen")
            if f_data:
                frozen_accs.append(f_data['metrics']['Average Accuracy'])
                frozen_forget.append(f_data['metrics']['Forgetting'])
                frozen_bwt.append(f_data['metrics'].get('Backward Transfer', 0.0))
            else:
                frozen_accs.append(0.0)
                frozen_forget.append(0.0)
                frozen_bwt.append(0.0)

            # Unfrozen
            u_data = grouped_results[s].get("Unfrozen")
            if u_data:
                unfrozen_accs.append(u_data['metrics']['Average Accuracy'])
                unfrozen_forget.append(u_data['metrics']['Forgetting'])
                unfrozen_bwt.append(u_data['metrics'].get('Backward Transfer', 0.0))
            else:
                unfrozen_accs.append(0.0)
                unfrozen_forget.append(0.0)
                unfrozen_bwt.append(0.0)

        # Plot Average Accuracy
        axes[0].bar(x - width/2, frozen_accs, width, label='Frozen', color='#3B82F6', alpha=0.85, edgecolor='black', linewidth=0.7)
        axes[0].bar(x + width/2, unfrozen_accs, width, label='Unfrozen', color='#EF4444', alpha=0.85, edgecolor='black', linewidth=0.7)
        axes[0].set_ylabel('Average IoU / Accuracy', fontsize=12)
        axes[0].set_title('Average Accuracy\n(Higher is Better)', fontsize=12, fontweight='bold', pad=10)
        axes[0].set_ylim([0, 1.0])
        axes[0].grid(axis='y', linestyle='--', alpha=0.5)
        axes[0].set_xticks(x)
        axes[0].set_xticklabels(labels, rotation=45, ha='right', fontsize=10)
        axes[0].legend()

        # Plot Forgetting
        axes[1].bar(x - width/2, frozen_forget, width, label='Frozen', color='#3B82F6', alpha=0.85, edgecolor='black', linewidth=0.7)
        axes[1].bar(x + width/2, unfrozen_forget, width, label='Unfrozen', color='#EF4444', alpha=0.85, edgecolor='black', linewidth=0.7)
        axes[1].set_ylabel('Forgetting Rate', fontsize=12)
        axes[1].set_title('Forgetting\n(Lower is Better)', fontsize=12, fontweight='bold', pad=10)
        max_forget = max(max(frozen_forget), max(unfrozen_forget)) if frozen_forget or unfrozen_forget else 0.1
        axes[1].set_ylim([0, max(0.1, max_forget * 1.2)])
        axes[1].grid(axis='y', linestyle='--', alpha=0.5)
        axes[1].set_xticks(x)
        axes[1].set_xticklabels(labels, rotation=45, ha='right', fontsize=10)
        axes[1].legend()

        # Plot Backward Transfer (BWT)
        axes[2].bar(x - width/2, frozen_bwt, width, label='Frozen', color='#3B82F6', alpha=0.85, edgecolor='black', linewidth=0.7)
        axes[2].bar(x + width/2, unfrozen_bwt, width, label='Unfrozen', color='#EF4444', alpha=0.85, edgecolor='black', linewidth=0.7)
        axes[2].set_ylabel('Backward Transfer (BWT)', fontsize=12)
        axes[2].set_title('Backward Transfer\n(Higher is Better)', fontsize=12, fontweight='bold', pad=10)
        axes[2].axhline(y=0, color='black', linestyle='-', linewidth=1.0)
        min_bwt = min(min(frozen_bwt), min(unfrozen_bwt)) if frozen_bwt or unfrozen_bwt else -0.1
        max_bwt = max(max(frozen_bwt), max(unfrozen_bwt)) if frozen_bwt or unfrozen_bwt else 0.1
        axes[2].set_ylim([min(-0.1, min_bwt * 1.2), max(0.1, max_bwt * 1.2)])
        axes[2].grid(axis='y', linestyle='--', alpha=0.5)
        axes[2].set_xticks(x)
        axes[2].set_xticklabels(labels, rotation=45, ha='right', fontsize=10)
        axes[2].legend()

    else:
        # Original single-bar plotting code
        strategies = []
        avg_accs = []
        forgettings = []
        bwts = []
        colors_bwt = []

        for name in sorted(strategy_results.keys()):
            results = strategy_results[name]
            label_key = clean_strategy_name(name)
            label = strategy_display_names.get(label_key, label_key.upper())
            strategies.append(label)
            avg_accs.append(results['metrics']['Average Accuracy'])
            forgettings.append(results['metrics']['Forgetting'])
            
            bwt = results['metrics'].get('Backward Transfer', 0.0)
            bwts.append(bwt)
            colors_bwt.append('green' if bwt >= 0 else 'red')

        # Plot Average Accuracy
        axes[0].bar(strategies, avg_accs, color='steelblue', alpha=0.85, edgecolor='black', linewidth=0.7)
        axes[0].set_ylabel('Average IoU / Accuracy', fontsize=12)
        axes[0].set_title('Average Accuracy\n(Higher is Better)', fontsize=12, fontweight='bold', pad=10)
        axes[0].set_ylim([0, 1.0])
        axes[0].grid(axis='y', linestyle='--', alpha=0.5)

        # Plot Forgetting
        axes[1].bar(strategies, forgettings, color='coral', alpha=0.85, edgecolor='black', linewidth=0.7)
        axes[1].set_ylabel('Forgetting Rate', fontsize=12)
        axes[1].set_title('Forgetting\n(Lower is Better)', fontsize=12, fontweight='bold', pad=10)
        max_forget = max(forgettings) if forgettings else 0.1
        axes[1].set_ylim([0, max(0.1, max_forget * 1.2)])
        axes[1].grid(axis='y', linestyle='--', alpha=0.5)

        # Plot Backward Transfer (BWT)
        axes[2].bar(strategies, bwts, color=colors_bwt, alpha=0.85, edgecolor='black', linewidth=0.7)
        axes[2].set_ylabel('Backward Transfer (BWT)', fontsize=12)
        axes[2].set_title('Backward Transfer\n(Higher is Better)', fontsize=12, fontweight='bold', pad=10)
        axes[2].axhline(y=0, color='black', linestyle='-', linewidth=1.0)
        min_bwt, max_bwt = min(bwts) if bwts else -0.1, max(bwts) if bwts else 0.1
        axes[2].set_ylim([min(-0.1, min_bwt * 1.2), max(0.1, max_bwt * 1.2)])
        axes[2].grid(axis='y', linestyle='--', alpha=0.5)

        for ax in axes:
            ax.set_xticks(range(len(strategies)))
            ax.set_xticklabels(strategies, rotation=45, ha='right', fontsize=10)

    for ax in axes:
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)

    plt.suptitle("Comparative Analysis of Continual Learning Strategies", fontsize=15, fontweight='bold', y=1.02)
    plt.tight_layout()

    save_path.parent.mkdir(parents=True, exist_ok=True)
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    print(f"\nSuccessfully generated comparison plot: {save_path}")

    print("\n" + "="*60)
    print("COMPARATIVE SUMMARY TABLE")
    print("="*60)
    print(f"| {'Strategy':<25} | {'Source Run':<12} | {'Backbone':<10} | {'Avg Accuracy':<12} | {'Forgetting':<10} | {'BWT':<10} |")
    print(f"|{'-'*27}|{'-'*14}|{'-'*12}|{'-'*14}|{'-'*12}|{'-'*12}|")
    for name, data in sorted(strategy_results.items()):
        metrics = data['metrics']
        avg_acc = metrics['Average Accuracy']
        forgetting = metrics['Forgetting']
        bwt = metrics.get('Backward Transfer', 0.0)
        run_name = data['run_dir']
        backbone_type = get_run_type(run_name, project_root)
        
        # Clean display name
        label_key = clean_strategy_name(name)
        disp_name = strategy_display_names.get(label_key, label_key.upper())
        
        print(f"| {disp_name:<25} | {run_name:<12} | {backbone_type:<10} | {avg_acc:<12.4f} | {forgetting:<10.4f} | {bwt:<10.4f} |")
    print("="*60 + "\n")


def plot_strategy_comparison(strategy_results: Dict[str, dict], save_path: str = None):
    from pathlib import Path
    save_path_p = Path(save_path) if save_path else Path("cl_comparison.png")
    project_root = Path("/content")
    plot_combined_comparison(strategy_results, save_path_p, project_root)


**Prediction Visualizer**

In [ ]:
def visualize_predictions(
    model: ContainerDefectSegmenter,
    dataset: ContainerDefectDataset,
    num_samples: int = 5,
    save_path: str = None
):
    """Visualize model predictions vs ground truth"""
    model.eval()
    device = next(model.parameters()).device
    indices = np.random.choice(len(dataset), num_samples, replace=False)

    fig, axes = plt.subplots(num_samples, 4, figsize=(16, 4 * num_samples))

    if num_samples == 1:
        axes = axes.reshape(1, -1)

    with torch.no_grad():
        for i, idx in enumerate(indices):
            image, mask, filename = dataset[idx]

            image_batch = image.unsqueeze(0).to(device)
            prediction = model.predict(image_batch).cpu().squeeze()

            mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
            std = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)
            image_denorm = image * std + mean
            image_denorm = torch.clamp(image_denorm, 0, 1)

            axes[i, 0].imshow(image_denorm.permute(1, 2, 0))
            axes[i, 0].set_title(f'Image\n{filename[:30]}...', fontsize=10)
            axes[i, 0].axis('off')

            axes[i, 1].imshow(mask.squeeze(), cmap='gray', vmin=0, vmax=1)
            axes[i, 1].set_title('Ground Truth', fontsize=10)
            axes[i, 1].axis('off')

            axes[i, 2].imshow(prediction, cmap='gray', vmin=0, vmax=1)
            axes[i, 2].set_title('Prediction', fontsize=10)
            axes[i, 2].axis('off')

            overlay = image_denorm.permute(1, 2, 0).clone()
            pred_mask = prediction > 0.5
            overlay[pred_mask, 0] = 1.0
            overlay[pred_mask, 1] = 0.0
            overlay[pred_mask, 2] = 0.0

            axes[i, 3].imshow(overlay)
            axes[i, 3].set_title('Overlay (Red = Defect)', fontsize=10)
            axes[i, 3].axis('off')

    plt.tight_layout()

    if save_path:
        plt.savefig(save_path, dpi=300, bbox_inches='tight')


**Generate Full Report**

In [ ]:
def generate_cl_report(strategy_results: Dict[str, dict], save_dir: str = SAVE_DIR):
    """Generate a complete analysis report with all visualizations"""
    save_dir = Path(save_dir)
    save_dir.mkdir(parents=True, exist_ok=True)

    plot_strategy_comparison(strategy_results, save_path=save_dir / "cl_comparison.png")

    for strategy_name, results in strategy_results.items():
        if results is not None:
            plot_accuracy_matrix(results, save_path=save_dir / f"cl_matrix_{strategy_name}.png")
            plot_task_performance_curves(results, save_path=save_dir / f"cl_curves_{strategy_name}.png")

    print("Report generation complete!")
    print(f"Figures saved to: {save_dir}")


In [ ]:
# Example: Visualize Results

# Load results from a single strategy
results_naive = load_cl_results('naive')

# Plot accuracy matrix
plot_accuracy_matrix(results_naive)

# Plot performance curves
plot_task_performance_curves(results_naive)

# Compare all strategies
all_results = {
    'naive': load_cl_results('naive'),
    'er': load_cl_results('er'),
    'der++': load_cl_results('der++')
}
plot_strategy_comparison(all_results)

# Generate full report
generate_cl_report(all_results)

# Visualize predictions
model = load_trained_model(f"{SAVE_DIR}/Naive_Fine-tuning_task3.pth")
val_dataset = ContainerDefectDataset(VALID_DIR, subset_size=50)
visualize_predictions(model, val_dataset, num_samples=5)